# 0. Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
from geopy.distance import distance
from shapely.geometry import LineString, Point
from pyproj import Transformer
import ast

from sklearn.cluster import DBSCAN

In [2]:
MAX_DISTANCE = 2 # miles
MAX_TIME_DIFF = 30 # minutes

TRIPS_PATH = '../data/simulated/trips.csv'
USERS_PATH = '../data/simulated/users.csv'
FRIENDS_PATH = '../data/simulated/friendships.csv'

# 1. Load Data

In [3]:
trips = pd.read_csv(TRIPS_PATH)
users = pd.read_csv(USERS_PATH)
friendships = pd.read_csv(FRIENDS_PATH)

print(f'Loaded {len(trips):,} trips across {trips["user_id"].nunique()} users')
print(f'Loaded {len(users)} user profiles')
print(f'Loaded {len(friendships)} friendship edges')

Loaded 2,181 trips across 35 users
Loaded 35 user profiles
Loaded 69 friendship edges


# 2. Friendship Parsing

In [4]:
def parse_friendships(friendships):
    bidirectional = pd.concat([
        friendships.rename(columns={'user_id_1': 'user', 'user_id_2': 'friend'}),
        friendships.rename(columns={'user_id_2': 'user', 'user_id_1': 'friend'})
    ])

    friendship_dict = bidirectional.groupby('user')['friend'].apply(set).to_dict()
    return friendship_dict

In [5]:
friendships_dict = parse_friendships(friendships)
friendships_dict

{0: {17, 25, 27, 32},
 1: {7, 9, 18, 19, 21, 30},
 2: {24},
 3: {17, 20, 26, 33},
 4: {15, 28},
 5: {18},
 6: {11, 14, 16, 34},
 7: {1, 8, 9, 28, 29},
 8: {7, 18, 26, 30, 31},
 9: {1, 7, 12, 13, 14, 17, 28},
 10: {23, 26, 27, 33},
 11: {6, 20, 25, 33},
 12: {9, 25, 26},
 13: {9, 14, 17, 28},
 14: {6, 9, 13, 17, 18, 26, 28},
 15: {4, 25},
 16: {6, 18, 25, 33},
 17: {0, 3, 9, 13, 14, 22, 28},
 18: {1, 5, 8, 14, 16, 19, 31},
 19: {1, 18, 23, 28},
 20: {3, 11},
 21: {1, 23, 27, 32},
 22: {17, 29},
 23: {10, 19, 21, 24},
 24: {2, 23},
 25: {0, 11, 12, 15, 16, 26, 27},
 26: {3, 8, 10, 12, 14, 25, 30},
 27: {0, 10, 21, 25},
 28: {4, 7, 9, 13, 14, 17, 19},
 29: {7, 22},
 30: {1, 8, 26},
 31: {8, 18},
 32: {0, 21},
 33: {3, 10, 11, 16},
 34: {6}}

# 3. Identify Recurring Trips

In [6]:
def get_location_clusters(trips, buffer_miles: int = 1, min_samples: int = 2) -> pd.DataFrame:
    trips = trips.copy()
    for col, lat_col, lon_col in [
        ('start_cluster', 'start_lat', 'start_lon'),
        ('end_cluster', 'end_lat', 'end_lon')
    ]:
        coords = trips[[lat_col, lon_col]].values
        coords_rad = np.radians(coords)
        
        # eps in radians: buffer_miles / earth_radius
        eps = buffer_miles / 3958.8
        
        labels = DBSCAN(eps=eps, min_samples=min_samples, algorithm='ball_tree', metric='haversine').fit_predict(coords_rad)
        trips[col] = labels
    
    return trips

def get_recurring_trips(trips, min_occurrences=2):
    trips = get_location_clusters(trips)

    # drop trips where either endpoint was an outlier
    trips = trips[
        (trips['start_cluster'] != -1) &
        (trips['end_cluster'] != -1)
    ]

    # count occurrences of each OD cluster pair
    od_counts = (
        trips
        .groupby(['start_cluster', 'end_cluster'])
        .size()
        .reset_index(name='count')
    )

    recurring_od = od_counts[od_counts['count'] >= min_occurrences][['start_cluster', 'end_cluster']]

    return trips.merge(recurring_od, on=['start_cluster', 'end_cluster'], how='inner')

In [19]:
my_trips = trips[trips['user_id'] == 19]
recurring_trips = get_recurring_trips(my_trips)
recurring_trips[['start_cluster', 'end_cluster']].drop_duplicates()

,start_cluster,end_cluster
0,0,0
1,1,1
3,1,2
4,2,1


In [21]:
import folium
import pandas as pd
import numpy as np

def visualize_recurring_trips(recurring_trips):
    # color palette for OD pairs
    colors = ['#377eb8','#984ea3','#4daf4a','#ff7f00',]

    center_lat = recurring_trips['start_lat'].mean()
    center_lon = recurring_trips['start_lon'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

    # assign a color to each unique OD cluster pair
    od_pairs = recurring_trips[['start_cluster', 'end_cluster']].drop_duplicates()
    od_pairs['color'] = [colors[i % len(colors)] for i in range(len(od_pairs))]
    recurring_trips = recurring_trips.merge(od_pairs, on=['start_cluster', 'end_cluster'])

    for _, trip in recurring_trips.iterrows():
        color = trip['color']
        label = f"OD: {trip['start_cluster']} → {trip['end_cluster']}"

        # start marker
        folium.CircleMarker(
            location=[trip['start_lat'], trip['start_lon']],
            radius=6, color=color, fill=True, fill_opacity=0.8,
            tooltip=f"Start | {label}"
        ).add_to(m)

        # end marker
        folium.CircleMarker(
            location=[trip['end_lat'], trip['end_lon']],
            radius=6, color=color, fill=True, fill_opacity=0.4,
            tooltip=f"End | {label}"
        ).add_to(m)

        # line connecting start to end
        folium.PolyLine(
            locations=[[trip['start_lat'], trip['start_lon']],
                       [trip['end_lat'], trip['end_lon']]],
            color=color, weight=2, opacity=0.5,
            tooltip=label
        ).add_to(m)

    return m

m = visualize_recurring_trips(recurring_trips)
m.save('recurring_trips.html')

# 4. Score Trips

In [24]:
def geodesic(a, b):
    dist_miles = distance(a, b).miles
    return dist_miles

def preprocess_trips(trips):
    #type conversion
    trips = trips.copy()
    trips['start_time'] = pd.to_datetime(trips['start_time'], format='mixed')
    trips['end_time'] = pd.to_datetime(trips['end_time'], format='mixed')
    if isinstance(trips['route_points'].iloc[0], str):
        trips['route_points'] = trips['route_points'].apply(ast.literal_eval)
    
    # feature engineering for recurring trip detection
    trips['time_bucket'] = trips['start_time'].dt.floor('30min').dt.time
    trips['day_of_week'] = trips['start_time'].dt.dayofweek
    return trips

def get_utm_transformer(lat, lon):
    # get the appropriate UTM zone transformer for a given location
    utm_zone = int((lon + 180) / 6) + 1
    hemisphere = "north" if lat >= 0 else "south"
    epsg = 32600 + utm_zone if hemisphere == "north" else 32700 + utm_zone
    return Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)

def project_and_create_polyline(route_points, transformer):
    route_proj = [transformer.transform(lon, lat) for lat, lon in route_points]
    return LineString(route_proj)

def get_pickup_begin_score(my_trip, other_trip, max_distance):
    # how close other trip's start is to my trip's start
    my_start = (my_trip['start_lat'], my_trip['start_lon'])
    other_start = (other_trip['start_lat'], other_trip['start_lon'])
    distance = geodesic(my_start, other_start)
    pickup_score = 1 - min(distance / max_distance, 1)
    return pickup_score

def get_pickup_enroute_score(my_route, other_trip, transformer, max_distance):
    # how close other trip's start is to any of my trip's route points
    METERS_TO_MILES = 0.000621371
    other_start = Point(transformer.transform(other_trip['start_lon'], other_trip['start_lat']))
    distance = other_start.distance(my_route) * METERS_TO_MILES
    pickup_score = 1 - min(distance / max_distance, 1)
    return pickup_score

def get_dropoff_end_score(my_trip, other_trip, max_distance):
    # how close other trip's end is to my trip's end
    my_end = (my_trip['end_lat'], my_trip['end_lon'])
    other_end = (other_trip['end_lat'], other_trip['end_lon'])
    distance = geodesic(my_end, other_end)
    dropoff_score = 1 - min(distance / max_distance, 1)
    return dropoff_score

def get_dropoff_enroute_score(my_route, other_trip, transformer, max_distance):
    # how close other trip's end is to any of my trip's route points
    METERS_TO_MILES = 0.000621371
    other_end = Point(transformer.transform(other_trip['end_lon'], other_trip['end_lat']))
    distance = other_end.distance(my_route) * METERS_TO_MILES
    dropoff_score = 1 - min(distance / max_distance, 1)
    return dropoff_score

def get_start_time_score(my_trip, other_trip, max_time_diff):
    time_diff = abs(my_trip['start_time'] - other_trip['start_time']).total_seconds() / 60
    time_score = 1 - min(time_diff / max_time_diff, 1)
    return time_score

def get_end_time_score(my_trip, other_trip, max_time_diff):
    time_diff = abs(my_trip['end_time'] - other_trip['end_time']).total_seconds() / 60
    time_score = 1 - min(time_diff / max_time_diff, 1)
    return time_score


def score_trips_for_user(user_id, trips, max_distance, max_time_diff, test=False, verbose=False):
    trips = preprocess_trips(trips)
    my_trips = trips[trips['user_id'] == user_id]
    my_recurring_trips = get_recurring_trips(my_trips)

    if my_recurring_trips.empty:
        print("No recurring trips found for user", user_id)
        return []

    my_friends = parse_friendships(friendships).get(user_id, set())
    friend_trips = [get_recurring_trips(trips[trips['user_id'] == friend]) for friend in my_friends]
    friend_trips = [df for df in friend_trips if not df.empty]
    other_recurring_trips = pd.concat(friend_trips) if friend_trips else pd.DataFrame()

    if other_recurring_trips.empty:
        print("No recurring trips found for friends of user", user_id)
        return []

    scores = []
    for _, my_trip in my_recurring_trips.iterrows():
        if verbose:
            print("Scoring trip:", my_trip['trip_id'])
            
        transformer = get_utm_transformer(my_trip['start_lat'], my_trip['start_lon'])
        my_route = project_and_create_polyline(my_trip['route_points'], transformer)

        for i, other_trip in other_recurring_trips.iterrows():
            pickup_begin_score = get_pickup_begin_score(my_trip, other_trip, max_distance)
            dropoff_end_score = get_dropoff_end_score(my_trip, other_trip, max_distance)
            
            pickup_enroute_score = get_pickup_enroute_score(my_route, other_trip, transformer, max_distance)
            if pickup_begin_score == 0 and pickup_enroute_score == 0:
                # no good pickup options, so skip this trip
                continue
            dropoff_enroute_score = get_dropoff_enroute_score(my_route, other_trip, transformer, max_distance)
            if dropoff_end_score == 0 and dropoff_enroute_score == 0:
                # no good dropoff options, so skip this trip
                continue

            start_time_score = get_start_time_score(my_trip, other_trip, max_time_diff)
            end_time_score = get_end_time_score(my_trip, other_trip, max_time_diff)

            # can pickup and dropoff at end, or pickup enroute and dropoff at end, or pickup at beginning and dropoff enroute, or pickup and dropoff enroute
            # time alignment should match with pickup/dropoff alignment - if pickup is enroute, then start time should be less important, if dropoff is enroute, then end time should be less important
            route_score = max(pickup_begin_score + dropoff_end_score + 0.5*start_time_score + 0.5*end_time_score,
                              pickup_enroute_score + dropoff_end_score + 0.25*start_time_score + 0.75*end_time_score,
                              pickup_begin_score + dropoff_enroute_score + 0.75*start_time_score + 0.25*end_time_score,
                              pickup_enroute_score + dropoff_enroute_score + 0.5*start_time_score + 0.5*end_time_score)

            
            total_score = route_score / 3
            scores.append({
                'user_id': user_id,
                'other_user': other_trip['user_id'],
                'my_trip': my_trip['trip_id'],
                'other_trip': other_trip['trip_id'],
                'score': round(total_score, 6)
            })

        if test:
            break
    
    scores.sort(key=lambda x: x['score'], reverse=True)
    return scores

In [ ]:
scores = {}
scores[0] = score_trips_for_user(0, trips, max_distance=MAX_DISTANCE, max_time_diff=MAX_TIME_DIFF, test=False)

In [25]:
all_scores = {}
for user_id in trips['user_id'].unique():
    user_id = int(user_id)
    print(f"Scoring trips for user {user_id}...")
    all_scores[user_id] = score_trips_for_user(user_id, trips, max_distance=MAX_DISTANCE, max_time_diff=MAX_TIME_DIFF)

Scoring trips for user 0...
Scoring trips for user 1...
Scoring trips for user 2...
Scoring trips for user 3...
Scoring trips for user 4...
Scoring trips for user 5...
Scoring trips for user 6...
Scoring trips for user 7...
Scoring trips for user 8...
Scoring trips for user 9...
Scoring trips for user 10...
Scoring trips for user 11...
Scoring trips for user 12...
Scoring trips for user 13...
Scoring trips for user 14...
Scoring trips for user 15...
Scoring trips for user 16...
Scoring trips for user 17...
Scoring trips for user 18...
Scoring trips for user 19...
Scoring trips for user 20...
Scoring trips for user 21...
Scoring trips for user 22...
Scoring trips for user 23...
Scoring trips for user 24...
Scoring trips for user 25...
Scoring trips for user 26...
Scoring trips for user 27...
Scoring trips for user 28...
Scoring trips for user 29...
Scoring trips for user 30...
Scoring trips for user 31...
Scoring trips for user 32...
Scoring trips for user 33...
Scoring trips for user 3

In [48]:
all_matches = [s for score in all_scores.values() for s in score if s['score'] > 0]
len(all_matches)

33576

In [49]:
matches_df = pd.DataFrame(all_matches)
matches_df.sort_values('score', ascending=False, inplace=True)
matches_df

,user_id,other_user,my_trip,other_trip,score
23330,21,1,1699,1653,0.913032
5161,9,1,1205,1186,0.892545
23331,21,1,1233,1187,0.886815
5162,9,1,494,475,0.884913
5163,9,1,575,554,0.877086
...,...,...,...,...,...
3888,6,14,96,114,0.031955
3905,6,14,96,2049,0.031955
3904,6,14,96,1840,0.031955
3890,6,14,96,423,0.031955


In [59]:
def visualize_top_matches(matches_df, trips_df, n_users=5):
    colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00',
          '#a65628', '#f781bf', '#66c2a5', '#ffd700', '#00ced1']
    top_matches = (
        matches_df
        .sort_values('score', ascending=False)
        .groupby('user_id')
        .first()
        .reset_index()
        .sort_values('score', ascending=False)
        .head(n_users)
        .reset_index(drop=True)
    )

    # look up lat/lon for each trip
    trip_locations = trips_df.set_index('trip_id')[['start_lat', 'start_lon', 'end_lat', 'end_lon']]

    center_lat = trips_df['start_lat'].mean()
    center_lon = trips_df['start_lon'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

    for i, match in top_matches.iterrows():
        color = colors[i]
        score = match['score']
        my_trip = trip_locations.loc[match['my_trip']]
        other_trip = trip_locations.loc[match['other_trip']]
        label = f"Match #{i+1} | Score: {score:.4f} | User {int(match['user_id'])} ↔ User {int(match['other_user'])}"

        # my trip start
        folium.CircleMarker(
            location=[my_trip['start_lat'], my_trip['start_lon']],
            radius=8, color=color, fill=True, fill_opacity=0.9,
            tooltip=f"{label} | My start"
        ).add_to(m)

        # my trip end
        folium.CircleMarker(
            location=[my_trip['end_lat'], my_trip['end_lon']],
            radius=8, color=color, fill=True, fill_opacity=0.4,
            tooltip=f"{label} | My end"
        ).add_to(m)

        # other trip start
        folium.CircleMarker(
            location=[other_trip['start_lat'], other_trip['start_lon']],
            radius=5, color=color, fill=True, fill_opacity=0.9,
            tooltip=f"{label} | Friend start"
        ).add_to(m)

        # other trip end
        folium.CircleMarker(
            location=[other_trip['end_lat'], other_trip['end_lon']],
            radius=5, color=color, fill=True, fill_opacity=0.4,
            tooltip=f"{label} | Friend end"
        ).add_to(m)

        # my trip route
        folium.PolyLine(
            locations=[[my_trip['start_lat'], my_trip['start_lon']],
                       [my_trip['end_lat'], my_trip['end_lon']]],
            color=color, weight=3, opacity=0.8, tooltip=label
        ).add_to(m)

        # other trip route (dashed)
        folium.PolyLine(
            locations=[[other_trip['start_lat'], other_trip['start_lon']],
                       [other_trip['end_lat'], other_trip['end_lon']]],
            color=color, weight=3, opacity=0.4, dash_array='10', tooltip=label
        ).add_to(m)

    return m

m = visualize_top_matches(matches_df, trips, 5)
m.save('top_matches.html')